<div style="
    width: 100%;
    box-sizing: border-box;
    margin: 20px 0;
    padding: 20px;
    background: #0b0f1a;
    border: 1px solid rgba(0,255,255,0.2);
    border-radius: 12px;
    color: #d6faff;
    font-family: Arial, sans-serif;
">

<h2 style="color:#00f5ff;">⚡ LLM Guardrails & Self-Correction Exercise</h2>

<p>
Build a professional roleplay chatbot representing a specific persona (<span style="color:#00f5ff;">Shakiru Sikiru</span>) by grounding the system prompt with context extracted from a <span style="color:#39ff14;">LinkedIn PDF profile</span> and summary text. 
The system must test the agent's ability to maintain a professional tone and adhere strictly to factual context.
</p>

<p>
Integrate a second LLM as an automated evaluator utilizing structured outputs via Pydantic to strictly validate the agent's response for:
<strong>is_acceptable (boolean verification) and feedback (detailed quality critique).</strong>
</p>

<p>
Implement an autonomous critique-and-refine workflow. If the evaluator flags a response as unacceptable (e.g., breaking character or failing safety/format constraints), the system must automatically <strong>trigger a rerun function that feeds the rejection reason back into the model for a corrected response.</strong>
</p>

</div>

In [1]:
# Import relevant libraries
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import os

In [2]:
# load environmental variables
load_dotenv(override=True)

True

In [3]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('NVIDIA_API_KEY_1')
# anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
deepseek_api_key = os.getenv('NVIDIA_API_KEY_2')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins nvapi-CG
Google API Key exists and begins AI
DeepSeek API Key exists and begins nva
Groq API Key exists and begins gsk_


In [4]:
openai = OpenAI(
    base_url = "https://integrate.api.nvidia.com/v1", 
    api_key = openai_api_key
)

In [5]:
# Read pdf file and extract text
reader = PdfReader("linkedin/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [6]:
print(linkedin)

   
Contact
1 Iyalode Close, Off Ijagemo Road,
Ijegun, Lagos State
2349068819192 (Mobile)
shakpro69@gmail.com
www.linkedin.com/in/shakirusikiru
(LinkedIn)
Top Skills
Natural Language Processing (NLP)
Large Language Models (LLM)
Computer Hardware Troubleshooting
Certifications
Python Developer
Introduction to SQL
Data Science
HTML5
The Data Science Course: Complete
Data science Bootcamp 2024
Shakiru Sikiru
Machine Learning Engineer || AI Engineer
Lagos State, Nigeria
Summary
Highly skilled Data Scientist with robust expertise in data analysis,
machine learning, and deep learning. Proven experience in
leveraging advanced statistical models and machine learning
algorithms to solve complex business problems and deliver
actionable insights. Proficient in Python, TensorFlow, and Keras,
with hands-on experience in predictive modeling, classification,
and clustering. Developed various data-driven projects, including
customer retention analysis, image classification using CNNs,
market segmentat

In [7]:
with open("linkedin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [8]:
name = "Shakiru Sikiru"

In [9]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [10]:
system_prompt

"You are acting as Shakiru Sikiru. You are answering questions on Shakiru Sikiru's website, particularly questions related to Shakiru Sikiru's career, background, skills and experience. Your responsibility is to represent Shakiru Sikiru for interactions on the website as faithfully as possible. You are given a summary of Shakiru Sikiru's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Shakiru Sikiru. I'm an software engineer, AI engineer, ML engineer and data scientist. I'm originally from Lagos, Nigeria, but I moved to Ogun state in 2025.\n\nI enjoy building machine learning models, exploring large language models, and solving real-world business problems. One of my favorite activities is turning complex datasets into useful insights, while continuously learning new AI techniques and im

In [11]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="openai/gpt-oss-120b", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [ ]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\Shakiru\anaconda3\envs\llm\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Shakiru\anaconda3\envs\llm\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Shakiru\anaconda3\envs\llm\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Shakiru\anaconda3\envs\llm\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_help

## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [14]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [15]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [16]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [30]:
import os
gemini = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [31]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [32]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="openai/gpt-oss-120b", messages=messages)
reply = response.choices[0].message.content

In [33]:
reply

'Hello! \n\nAt this time I don’t hold any issued patents. My focus has been on building and deploying machine‑learning models, developing data‑driven solutions for businesses, and continuously learning new AI techniques. That said, I’m always exploring innovative approaches—especially in natural language processing and large‑language‑model applications—and I’m open to collaborating on projects that could lead to patentable inventions in the future. \n\nIf you have a particular problem you think could benefit from a novel solution, I’d love to discuss how we might turn that idea into a real‑world impact (and possibly a patent). Feel free to reach out!'

In [34]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The agent correctly states that Shakiru Sikiru does not hold any patents, which aligns with the provided context. The response also goes further to maintain the professional and engaging persona by discussing current focuses, future aspirations in innovation, and inviting collaboration, all of which are relevant to Shakiru's profile.")

In [35]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="openai/gpt-oss-120b", messages=messages)
    return response.choices[0].message.content

In [36]:
def chat(message, history):
    if "patent" not in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="openai/gpt-oss-120b", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [38]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
